# Kayak Project — Weather Data Collection

This notebook gets the weather of the project's 35 French destinations, from the OpenWeather API. It is the first step of the pipeline: its score ranks the destinations, and its clean file feeds the next notebooks. One destination, Gorges du Verdon, gets no coordinates from the geocoder, so 34 destinations have weather data.

Steps:
- List the destinations
- Get their coordinates (geocoding)
- Get the weather forecasts
- Average them per city and compute a weather score
- Save the clean file

The next cell imports the libraries.

In [1]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

## API Configuration

Loads the OpenWeather API key from the `.env` file. The key stays out of the code, because the code is shared. No output.

In [2]:
load_dotenv()

OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
assert OPENWEATHER_API_KEY is not None, "Missing OPENWEATHER_API_KEY in .env"

## City List

Turns the list of 35 destinations into a table and gives each one a `city_id`. The brief asks for this unique identifier: it later links each hotel to its city. The output shows the first 5 rows, from 1 Mont Saint Michel to 5 Rouen.

In [3]:
cities = [
    "Mont Saint Michel",
    "St Malo",
    "Bayeux",
    "Le Havre",
    "Rouen",
    "Paris",
    "Amiens",
    "Lille",
    "Strasbourg",
    "Chateau du Haut Koenigsbourg",
    "Colmar",
    "Eguisheim",
    "Besancon",
    "Dijon",
    "Annecy",
    "Grenoble",
    "Lyon",
    "Gorges du Verdon",
    "Bormes les Mimosas",
    "Cassis",
    "Marseille",
    "Aix en Provence",
    "Avignon",
    "Uzes",
    "Nimes",
    "Aigues Mortes",
    "Saintes Maries de la mer",
    "Collioure",
    "Carcassonne",
    "Ariege",
    "Toulouse",
    "Montauban",
    "Biarritz",
    "Bayonne",
    "La Rochelle"
]

cities_df = pd.DataFrame({
    "city_id": range(1, len(cities) + 1),
    "city_name": cities
})

cities_df.head()

,city_id,city_name
0,1,Mont Saint Michel
1,2,St Malo
2,3,Bayeux
3,4,Le Havre
4,5,Rouen


## Geocoding

Asks the OpenWeather Geocoding API for the latitude and longitude of each destination, because the forecast API works with coordinates, not names. It waits 1 second between calls. The output shows 34 destinations with coordinates. Gorges du Verdon stays empty: the geocoder found nothing for it.

In [4]:
def geocode_city(city_name, country="FR"):
    url = "https://api.openweathermap.org/geo/1.0/direct"
    params = {
        "q": f"{city_name},{country}",
        "limit": 1,
        "appid": OPENWEATHER_API_KEY
    }
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    if not data:
        return {
            "latitude": None,
            "longitude": None,
        }

    return {
        "latitude": data[0]["lat"],
        "longitude": data[0]["lon"],
        "display_name": data[0].get("name")
    }

geocoded_rows = []

for city in cities_df["city_name"]:
    try:
        result = geocode_city(city)
        geocoded_rows.append(result)
    except Exception as e:
        geocoded_rows.append({
            "latitude": None,
            "longitude": None,
            "display_name": None
        })
        print(f"Geocoding failed for {city}: {e}")

    time.sleep(1)

geo_df = pd.concat([cities_df, pd.DataFrame(geocoded_rows)], axis=1)

geo_df

,city_id,city_name,latitude,longitude,display_name
0,1,Mont Saint Michel,48.635954,-1.511460,Mont Saint-Michel
1,2,St Malo,48.649518,-2.026041,Saint-Malo
2,3,Bayeux,49.276462,-0.702474,Bayeux
3,4,Le Havre,49.493898,0.107973,Le Havre
4,5,Rouen,49.440459,1.093966,Rouen
5,6,Paris,48.858890,2.320041,Paris
6,7,Amiens,49.894171,2.295695,Amiens
7,8,Lille,50.636565,3.063528,Lille
8,9,Strasbourg,48.584614,7.750713,Strasbourg
9,10,Chateau du Haut Koenigsbourg,48.249523,7.345492,Château du Haut-Kœnigsbourg


## Raw Data Storage

Saves the geocoded table to `data/raw/cities_geocoded.csv`. This raw file keeps a trace of the collection, and notebook 03 reads `city_id` from it. No output.

In [5]:
import os
os.makedirs("../data/raw", exist_ok=True)
geo_df.to_csv("../data/raw/cities_geocoded.csv", index=False)

## Weather Retrieval

Defines the function that calls the OpenWeather Forecast API for one pair of coordinates. This API covers 5 days, while the brief asks for 7: the README documents this difference. No output: the function is only defined here.

In [6]:
def get_weather(lat, lon):

    url = "https://api.openweathermap.org/data/2.5/forecast"

    params = {
        "lat": lat,
        "lon": lon,
        "units": "metric",
        "appid": OPENWEATHER_API_KEY
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    return response.json()

## Weather Data Collection

Calls the forecast for each destination and keeps one row per time step, with temperature, humidity, wind speed and rain probability. A destination without coordinates gets no request: the output says so for Gorges du Verdon. Then it shows the first rows for Mont Saint Michel, with 13.64 °C on the first one.

In [7]:
weather_rows = []

for _, row in geo_df.iterrows():

    city = row["city_name"]
    lat = row["latitude"]
    lon = row["longitude"]

    if pd.isna(lat) or pd.isna(lon):
        print(city, "has no coordinates: no weather request")
        continue

    try:
        weather = get_weather(lat, lon)

        for item in weather["list"]:

            weather_rows.append({
                "city": city,
                "latitude": lat,
                "longitude": lon,
                "temperature": item["main"]["temp"],
                "humidity": item["main"]["humidity"],
                "wind_speed": item["wind"]["speed"],
                "rain_prob": item["pop"]
            })

    except requests.HTTPError as e:
        print(city, 'HTTP', e.response.status_code)

weather_df = pd.DataFrame(weather_rows)
weather_df.head()

Gorges du Verdon has no coordinates: no weather request


,city,latitude,longitude,temperature,humidity,wind_speed,rain_prob
0,Mont Saint Michel,48.635954,-1.51146,13.64,72,2.24,0.0
1,Mont Saint Michel,48.635954,-1.51146,13.81,71,2.42,0.0
2,Mont Saint Michel,48.635954,-1.51146,11.52,83,2.87,0.0
3,Mont Saint Michel,48.635954,-1.51146,7.96,91,2.06,0.0
4,Mont Saint Michel,48.635954,-1.51146,6.60,95,1.97,0.0


## Aggregation

Averages each weather measure per destination, to get one line per city. No output. The result covers 34 destinations.

In [8]:
city_weather = (
    weather_df
    .groupby(["city","latitude","longitude"])
    .agg({
        "temperature":"mean",
        "humidity":"mean",
        "rain_prob":"mean",
        "wind_speed":"mean"
    })
    .reset_index()
)

## Weather Score

Computes the score that ranks the destinations:

weather_score = temperature - rain_prob * 10 - wind_speed

A higher temperature raises the score. Rain and wind lower it. `rain_prob` goes from 0 to 1, so the factor 10 puts it on the scale of degrees: 10% of rain risk counts as one degree less. These weights are a choice. Humidity is collected but not used in the score. No output.

In [9]:
city_weather["weather_score"] = (
    city_weather["temperature"]
    - city_weather["rain_prob"] * 10
    - city_weather["wind_speed"]
)

Sorts the destinations from the highest score to the lowest. No output.

In [10]:
city_weather = city_weather.sort_values("weather_score", ascending=False)

## Top 5

Shows the 5 best destinations: Bormes les Mimosas (7.82), Bayonne (7.62), Biarritz (7.62), Cassis (7.57) and Collioure (7.12). Bayonne and Biarritz are only 0.001 apart. This ranking depends on the forecasts of 31 October 2026, the day of the collection.

In [11]:
top5 = city_weather.head(5)
top5

,city,latitude,longitude,temperature,humidity,rain_prob,wind_speed,weather_score
10,Bormes les Mimosas,43.150697,6.341928,12.19925,70.775,0.16550,2.72550,7.81875
7,Bayonne,43.494514,-1.473666,11.36525,81.250,0.06450,3.10000,7.62025
9,Biarritz,43.483252,-1.559278,12.09100,78.375,0.06375,3.83425,7.61925
12,Cassis,43.214036,5.539632,11.96275,77.450,0.17775,2.61600,7.56925
14,Collioure,42.525050,3.083155,12.54800,68.400,0.15625,3.86475,7.12075


Saves the 34 destinations to `data/clean/cities_weather.csv`, the input of notebook 02. No output.

In [12]:
city_weather.to_csv("../data/clean/cities_weather.csv", index=False)